In [ ]:
!pip install nba_api
import requests
import pandas as pd
from datetime import datetime
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display, HTML
import time

# Fetch today's games from NBA Stats API (JSON format)
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()

        today = datetime.now().strftime('%Y-%m-%d')
        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],  # Home team abbreviation
                        'away_team': game['v']['ta'],  # Away team abbreviation
                        'home_team_id': game['h']['tid'],  # Home team ID
                        'away_team_id': game['v']['tid']   # Away team ID
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season='2024-25'):
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Fetching game logs for player ID: {player_id}")
            game_logs = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season').get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying game logs fetch for player {player_id} (Attempt {attempt + 1})...")
                time.sleep(5)
            else:
                print(f"Failed to fetch game logs for player {player_id}: {e}")
                return pd.DataFrame()

# Calculate per-player stats (Points, Rebounds, Assists)
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    stat_values = pd.Series(stat_values).fillna(0)  # Fill NaN values with zero
    mean = round(stat_values.mean(), 2)
    std_dev = round(stat_values.std(), 2)
    cv = round(std_dev / mean, 2) if mean != 0 else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']
        home_team = game['home_team']
        away_team = game['away_team']

        print(f"Processing teams: {home_team} vs {away_team}")

        # Fetch player stats for both teams
        for team_id, team_name in [(home_team_id, home_team), (away_team_id, away_team)]:
            print(f"Processing team: {team_name}")
            try:
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                if not team_roster.empty:
                    for _, player in team_roster.iterrows():
                        try:
                            player_id = player['PLAYER_ID']
                            player_name = player['PLAYER']
                            print(f"Processing player: {player_name} (ID: {player_id})")

                            game_logs = fetch_player_game_logs(player_id)

                            if game_logs.empty:
                                continue

                            ppg_values = game_logs['PTS']
                            rpg_values = game_logs['REB']
                            apg_values = game_logs['AST']

                            mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                            mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                            mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                            all_players_data.append({
                                'Player': player_name,
                                'Team': team_name,
                                'PPG': mean_ppg,
                                'PPG SD': std_dev_ppg,
                                'PPG CV': cv_ppg,
                                'RPG': mean_rpg,
                                'RPG SD': std_dev_rpg,
                                'RPG CV': cv_rpg,
                                'APG': mean_apg,
                                'APG SD': std_dev_apg,
                                'APG CV': cv_apg
                            })
                            time.sleep(1)
                        except Exception as e:
                            print(f"Error processing player {player_name}: {e}")
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    return pd.DataFrame(all_players_data)

# Display results as an HTML table in Jupyter Notebook
def display_results(df, stat_type):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)

    if stat_type == 'Points':
        sorted_df = df.sort_values('PPG CV', ascending=True)
        print("\nTop Players by Points CV:")
        display(HTML(sorted_df[['Player', 'Team', 'PPG', 'PPG SD', 'PPG CV']].to_html(index=False, justify='center')))
    elif stat_type == 'Rebounds':
        sorted_df = df.sort_values('RPG CV', ascending=True)
        print("\nTop Players by Rebounds CV:")
        display(HTML(sorted_df[['Player', 'Team', 'RPG', 'RPG SD', 'RPG CV']].to_html(index=False, justify='center')))
    elif stat_type == 'Assists':
        sorted_df = df.sort_values('APG CV', ascending=True)
        print("\nTop Players by Assists CV:")
        display(HTML(sorted_df[['Player', 'Team', 'APG', 'APG SD', 'APG CV']].to_html(index=False, justify='center')))

# Run the analysis and display the output
if __name__ == "__main__":
    print("Fetching data for today's games...")
    stats_data = analyze_teams_for_today()

    if not stats_data.empty:
        display_results(stats_data, 'Points')
        display_results(stats_data, 'Rebounds')
        display_results(stats_data, 'Assists')
    else:
        print("No games or data available for today.")


Fetching data for today's games...
Processing teams: CHA vs NYK
Processing team: CHA
Processing player: Miles Bridges (ID: 1628970)
Fetching game logs for player ID: 1628970
Processing player: LaMelo Ball (ID: 1630163)
Fetching game logs for player ID: 1630163
Processing player: Grant Williams (ID: 1629684)
Fetching game logs for player ID: 1629684
Processing player: DaQuan Jeffries (ID: 1629610)
Fetching game logs for player ID: 1629610
Processing player: Nick Richards (ID: 1630208)
Fetching game logs for player ID: 1630208
Processing player: Mark Williams (ID: 1631109)
Fetching game logs for player ID: 1631109
Processing player: Nick Smith Jr. (ID: 1641733)
Fetching game logs for player ID: 1641733
Processing player: Josh Green (ID: 1630182)
Fetching game logs for player ID: 1630182
Processing player: Cody Martin (ID: 1628998)
Fetching game logs for player ID: 1628998
Processing player: Moussa Diabate (ID: 1631217)
Fetching game logs for player ID: 1631217
Processing player: Vasilije

KeyboardInterrupt: 

In [ ]:
# Required Libraries
!pip install nba_api
import requests
import pandas as pd
from datetime import datetime
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from IPython.display import display, HTML
import time

# Fetch today's games from NBA Stats API (JSON format)
def fetch_schedule():
    url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
    response = requests.get(url)
    data = response.json()

    # Get today's date in the format used in the JSON
    today = datetime.now().strftime('%Y-%m-%d')

    # Extract games for today
    games_today = []
    for month_data in data['lscd']:  # Loop through each month's schedule
        for game in month_data['mscd']['g']:  # Extract each game
            if game['gdte'] == today:  # Match today's date
                games_today.append(game)

    return games_today

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season='2024-25'):
    retries = 3
    for attempt in range(retries):
        try:
            game_logs = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season').get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
            else:
                return pd.DataFrame()

# Calculate statistics for a list of values
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    mean = sum(stat_values) / len(stat_values)
    std_dev = pd.Series(stat_values).std()
    cv = std_dev / mean if mean != 0 else None

    # Ensure that None values are handled before rounding
    mean = round(mean, 2) if mean is not None else None
    std_dev = round(std_dev, 2) if std_dev is not None else None
    cv = round(cv, 2) if cv is not None else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['h']['tid']  # Home team ID
        away_team_id = game['v']['tid']  # Away team ID
        home_team = game['h']['ta']  # Home team abbreviation
        away_team = game['v']['ta']  # Away team abbreviation

        for team_id, team_name in [(home_team_id, home_team), (away_team_id, away_team)]:
            team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

            if not team_roster.empty:
                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    position = player['POSITION']

                    game_logs = fetch_player_game_logs(player_id)

                    if game_logs.empty:
                        continue

                    ppg_values = game_logs['PTS'].tolist()
                    rpg_values = game_logs['REB'].tolist()
                    apg_values = game_logs['AST'].tolist()

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                    all_players_data.append({
                        'Player': f"{player_name} ({position})",
                        'Team': team_name,
                        'PPG': mean_ppg, 'PPG SD': std_dev_ppg, 'PPG CV': cv_ppg,
                        'RPG': mean_rpg, 'RPG SD': std_dev_rpg, 'RPG CV': cv_rpg,
                        'APG': mean_apg, 'APG SD': std_dev_apg, 'APG CV': cv_apg
                    })

    return pd.DataFrame(all_players_data)

# Display results sorted by consistency (lowest CV first)
def display_sorted_stats(df, category):
    if category == 'Points':
        stats = df[['Player', 'Team', 'PPG', 'PPG SD', 'PPG CV']].sort_values(by='PPG CV').dropna()
    elif category == 'Rebounds':
        stats = df[['Player', 'Team', 'RPG', 'RPG SD', 'RPG CV']].sort_values(by='RPG CV').dropna()
    elif category == 'Assists':
        stats = df[['Player', 'Team', 'APG', 'APG SD', 'APG CV']].sort_values(by='APG CV').dropna()
    else:
        print(f"Unknown category: {category}")
        return

    print(f"\nTop Players by {category} Consistency (Lowest CV):")
    display(HTML(stats.to_html(index=False)))

# Run the analysis
if __name__ == "__main__":
    print("Fetching data for today's games...")
    stats_data = analyze_teams_for_today()

    if not stats_data.empty:
        display_sorted_stats(stats_data, 'Points')
        display_sorted_stats(stats_data, 'Rebounds')
        display_sorted_stats(stats_data, 'Assists')
    else:
        print("No games or data available for today.")


Fetching data for today's games...
No games scheduled for today.
No games or data available for today.


In [ ]:
!pip install nba_api openpyxl
import requests
import pandas as pd
from datetime import datetime
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display, HTML
import os
import time

# Fetch today's games from NBA Stats API (JSON format)
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()

        today = datetime.now().strftime('%Y-%m-%d')
        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],  # Home team abbreviation
                        'away_team': game['v']['ta'],  # Away team abbreviation
                        'home_team_id': game['h']['tid'],  # Home team ID
                        'away_team_id': game['v']['tid']   # Away team ID
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season='2024-25'):
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Fetching game logs for player ID: {player_id}")
            game_logs = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season').get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying game logs fetch for player {player_id} (Attempt {attempt + 1})...")
                time.sleep(5)
            else:
                print(f"Failed to fetch game logs for player {player_id}: {e}")
                return pd.DataFrame()

# Calculate per-player stats (Points, Rebounds, Assists)
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    stat_values = pd.Series(stat_values).fillna(0)  # Fill NaN values with zero
    mean = round(stat_values.mean(), 2)
    std_dev = round(stat_values.std(), 2)
    cv = round(std_dev / mean, 2) if mean != 0 else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']
        home_team = game['home_team']
        away_team = game['away_team']

        print(f"Processing teams: {home_team} vs {away_team}")

        # Fetch player stats for both teams
        for team_id, team_name in [(home_team_id, home_team), (away_team_id, away_team)]:
            print(f"Processing team: {team_name}")
            try:
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                if not team_roster.empty:
                    for _, player in team_roster.iterrows():
                        try:
                            player_id = player['PLAYER_ID']
                            player_name = player['PLAYER']
                            player_position = player['POSITION']  # Fetch player's position
                            print(f"Processing player: {player_name} (ID: {player_id})")

                            game_logs = fetch_player_game_logs(player_id)

                            if game_logs.empty:
                                continue

                            ppg_values = game_logs['PTS']
                            rpg_values = game_logs['REB']
                            apg_values = game_logs['AST']

                            mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                            mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                            mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                            all_players_data.append({
                                'Player': player_name,
                                'Position': player_position,
                                'Team': team_name,
                                'PPG': mean_ppg,
                                'PPG SD': std_dev_ppg,
                                'PPG CV': cv_ppg,
                                'RPG': mean_rpg,
                                'RPG SD': std_dev_rpg,
                                'RPG CV': cv_rpg,
                                'APG': mean_apg,
                                'APG SD': std_dev_apg,
                                'APG CV': cv_apg
                            })
                            time.sleep(1)
                        except Exception as e:
                            print(f"Error processing player {player_name}: {e}")
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    return pd.DataFrame(all_players_data)

# Display results as an HTML table in Jupyter Notebook
def display_results(df, stat_type):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)

    if stat_type == 'Points':
        sorted_df = df.sort_values('PPG CV', ascending=True)
        print("\nTop Players by Points CV:")
        display(HTML(sorted_df[['Player', 'Position', 'Team', 'PPG', 'PPG SD', 'PPG CV']].to_html(index=False, justify='center')))
    elif stat_type == 'Rebounds':
        sorted_df = df.sort_values('RPG CV', ascending=True)
        print("\nTop Players by Rebounds CV:")
        display(HTML(sorted_df[['Player', 'Position', 'Team', 'RPG', 'RPG SD', 'RPG CV']].to_html(index=False, justify='center')))
    elif stat_type == 'Assists':
        sorted_df = df.sort_values('APG CV', ascending=True)
        print("\nTop Players by Assists CV:")
        display(HTML(sorted_df[['Player', 'Position', 'Team', 'APG', 'APG SD', 'APG CV']].to_html(index=False, justify='center')))

# Run the analysis and save the output to a spreadsheet
if __name__ == "__main__":
    print("Fetching data for today's games...")
    stats_data = analyze_teams_for_today()

    if not stats_data.empty:
        display_results(stats_data, 'Points')
        display_results(stats_data, 'Rebounds')
        display_results(stats_data, 'Assists')

        # Save the output to an Excel file
        try:
            desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")  # Get desktop path
            file_path = os.path.join(desktop_path, "NBA_Stats_Today.xlsx")

            # Write to Excel
            stats_data.to_excel(file_path, index=False, sheet_name='NBA Stats')

            print(f"Results saved to {file_path}")
        except Exception as e:
            print(f"Error saving to Excel: {e}")
    else:
        print("No games or data available for today.")


Fetching data for today's games...
No games scheduled for 2024-11-28.
No games scheduled for today.
No games or data available for today.


In [ ]:
!pip install nba_api openpyxl
import requests
import pandas as pd
from datetime import datetime
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display, HTML
import os
import time

# Fetch today's games from NBA Stats API (JSON format)
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        today = datetime.now().strftime('%Y-%m-%d')
        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season='2024-25'):
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Fetching game logs for player ID: {player_id}")
            game_logs = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season').get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying game logs fetch for player {player_id} (Attempt {attempt + 1})...")
                time.sleep(5)
            else:
                print(f"Failed to fetch game logs for player {player_id}: {e}")
                return pd.DataFrame()

# Calculate per-player stats (Points, Rebounds, Assists)
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    stat_values = pd.Series(stat_values).fillna(0)
    mean = round(stat_values.mean(), 2)
    std_dev = round(stat_values.std(), 2)
    cv = round(std_dev / mean, 2) if mean != 0 else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']
        home_team = game['home_team']
        away_team = game['away_team']

        print(f"Processing teams: {home_team} vs {away_team}")

        for team_id, team_name in [(home_team_id, home_team), (away_team_id, away_team)]:
            print(f"Processing team: {team_name}")
            try:
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                if not team_roster.empty:
                    for _, player in team_roster.iterrows():
                        try:
                            player_id = player['PLAYER_ID']
                            player_name = player['PLAYER']
                            player_position = player['POSITION']
                            print(f"Processing player: {player_name} (ID: {player_id})")

                            game_logs = fetch_player_game_logs(player_id)

                            if game_logs.empty:
                                continue

                            ppg_values = game_logs['PTS']
                            rpg_values = game_logs['REB']
                            apg_values = game_logs['AST']

                            mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                            mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                            mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                            all_players_data.append({
                                'Player': player_name,
                                'Position': player_position,
                                'Team': team_name,
                                'PPG': mean_ppg,
                                'PPG SD': std_dev_ppg,
                                'PPG CV': cv_ppg,
                                'RPG': mean_rpg,
                                'RPG SD': std_dev_rpg,
                                'RPG CV': cv_rpg,
                                'APG': mean_apg,
                                'APG SD': std_dev_apg,
                                'APG CV': cv_apg
                            })
                            time.sleep(1)
                        except Exception as e:
                            print(f"Error processing player {player_name}: {e}")
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    return pd.DataFrame(all_players_data)

# Save results to Excel
def save_to_excel(df):
    try:
        desktop_path = r'C:\Users\MaterialsLab\Desktop\nba_predictions.xlsx'  # Specify the path to your desktop
        file_path = os.path.join(desktop_path, "nba_predictions.xlsx")

        # Create separate DataFrames for each stat type
        points_df = df[['Player', 'Position', 'Team', 'PPG', 'PPG SD', 'PPG CV']].sort_values('PPG CV')
        rebounds_df = df[['Player', 'Position', 'Team', 'RPG', 'RPG SD', 'RPG CV']].sort_values('RPG CV')
        assists_df = df[['Player', 'Position', 'Team', 'APG', 'APG SD', 'APG CV']].sort_values('APG CV')

        # Write to Excel
        with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
            points_df.to_excel(writer, index=False, sheet_name='Points Consistency')
            rebounds_df.to_excel(writer, index=False, sheet_name='Rebounds Consistency')
            assists_df.to_excel(writer, index=False, sheet_name='Assists Consistency')

        print(f"Results saved to C:\\Users\\MaterialsLab\\Desktop\\nba_predictions.xlsx")
    except Exception as e:
        print(f"Error saving to Excel: {e}")

# Run the analysis and save the output
if __name__ == "__main__":
    print("Fetching data for today's games...")
    stats_data = analyze_teams_for_today()

    if not stats_data.empty:
        save_to_excel(stats_data)
    else:
        print("No games or data available for today.")


Fetching data for today's games...
Processing teams: CHA vs NYK
Processing team: CHA
Processing player: Miles Bridges (ID: 1628970)
Fetching game logs for player ID: 1628970
Processing player: LaMelo Ball (ID: 1630163)
Fetching game logs for player ID: 1630163
Processing player: Grant Williams (ID: 1629684)
Fetching game logs for player ID: 1629684
Processing player: DaQuan Jeffries (ID: 1629610)
Fetching game logs for player ID: 1629610
Processing player: Nick Richards (ID: 1630208)
Fetching game logs for player ID: 1630208
Processing player: Mark Williams (ID: 1631109)
Fetching game logs for player ID: 1631109
Processing player: Nick Smith Jr. (ID: 1641733)
Fetching game logs for player ID: 1641733
Processing player: Josh Green (ID: 1630182)
Fetching game logs for player ID: 1630182
Processing player: Cody Martin (ID: 1628998)
Fetching game logs for player ID: 1628998
Processing player: Moussa Diabate (ID: 1631217)
Fetching game logs for player ID: 1631217
Processing player: Vasilije

In [ ]:
import requests
from datetime import datetime

# Fetch today's games from NBA Stats API (JSON format)
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()

        today = datetime.now().strftime('%Y-%m-%d')
        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],  # Home team abbreviation
                        'away_team': game['v']['ta'],  # Away team abbreviation
                        'home_team_id': game['h']['tid'],  # Home team ID
                        'away_team_id': game['v']['tid'],  # Away team ID
                        'game_id': game['gid']  # Game ID
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Test the schedule fetching function
games = fetch_schedule()
if games:
    print(f"Games scheduled for today ({datetime.now().strftime('%Y-%m-%d')}):")
    for game in games:
        print(f"{game['away_team']} vs {game['home_team']} (Game ID: {game['game_id']})")
else:
    print("No games found or an error occurred.")

Games scheduled for today (2024-11-29):
NYK vs CHA (Game ID: 0022400040)
CLE vs ATL (Game ID: 0022400041)
NOP vs MEM (Game ID: 0022400042)
ORL vs BKN (Game ID: 0022400043)
LAC vs MIN (Game ID: 0022400044)
DET vs IND (Game ID: 0022400045)
TOR vs MIA (Game ID: 0022400046)
BOS vs CHI (Game ID: 0022400047)
OKC vs LAL (Game ID: 0022400048)
SAC vs POR (Game ID: 0022400049)


In [ ]:
# Install the necessary package for NBA data
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
import pandas as pd

# Fetch the data
team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(season="2024-25", season_type="Regular Season", league_id="00")

# Parse the response to a pandas DataFrame
data = team_metrics.get_data_frames()[0]  # Extract the first DataFrame (this is where our data resides)

# Filter the required columns
columns_of_interest = ["TEAM_NAME", "E_OFF_RATING", "E_DEF_RATING", "E_PACE"]

# Create a new DataFrame with only the columns we care about
filtered_data = data[columns_of_interest]

# Display the filtered data
print(filtered_data)



In [ ]:
!pip install nba_api pytz

import json
import pandas as pd
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import playergamelog, commonteamroster
from datetime import datetime
import requests
import time
import pytz

# Define EST timezone
EST = pytz.timezone('US/Eastern')

# Team abbreviation to full team name mapping
team_abbr_to_full = {
    'CHA': 'Charlotte Hornets', 'MIA': 'Miami Heat', 'CLE': 'Cleveland Cavaliers', 'ATL': 'Atlanta Hawks',
    'IND': 'Indiana Pacers', 'POR': 'Portland Trail Blazers', 'ORL': 'Orlando Magic', 'CHI': 'Chicago Bulls',
    'PHI': 'Philadelphia 76ers', 'HOU': 'Houston Rockets', 'WAS': 'Washington Wizards', 'LAC': 'Los Angeles Clippers',
    'DAL': 'Dallas Mavericks', 'NYK': 'New York Knicks', 'MEM': 'Memphis Grizzlies', 'DET': 'Detroit Pistons',
    'MIN': 'Minnesota Timberwolves', 'SAC': 'Sacramento Kings', 'NOP': 'New Orleans Pelicans', 'TOR': 'Toronto Raptors',
    'SAS': 'San Antonio Spurs', 'LAL': 'Los Angeles Lakers', 'PHX': 'Phoenix Suns', 'BKN': 'Brooklyn Nets',
    'UTA': 'Utah Jazz', 'DEN': 'Denver Nuggets', 'GSW': 'Golden State Warriors', 'OKC': 'Oklahoma City Thunder'
}

# Fetch the data for team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(season="2024-25", season_type="Regular Season", league_id="00")
        data = team_metrics.get_data_frames()[0]  # Extract the first DataFrame (this is where our data resides)
        columns_of_interest = ["TEAM_NAME", "E_OFF_RATING", "E_DEF_RATING", "E_PACE"]
        filtered_data = data[columns_of_interest]
        return filtered_data
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return pd.DataFrame()

# Fetch today's NBA games from NBA Stats API
def fetch_schedule():
    try:
        # Get the current date in EST
        now = datetime.now(EST)
        today = now.strftime('%Y-%m-%d')  # Format the date as YYYY-MM-DD
        print(f"Fetching schedule for {today} (EST)")

        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()

        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],  # Home team abbreviation
                        'away_team': game['v']['ta'],  # Away team abbreviation
                        'home_team_id': game['h']['tid'],  # Home team ID
                        'away_team_id': game['v']['tid']   # Away team ID
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season='2024-25'):
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Fetching game logs for player ID: {player_id}")
            game_logs = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season').get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying game logs fetch for player {player_id} (Attempt {attempt + 1})...")
                time.sleep(5)
            else:
                print(f"Failed to fetch game logs for player {player_id}: {e}")
                return pd.DataFrame()

# Calculate per-player stats (Points, Rebounds, Assists)
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    stat_values = pd.Series(stat_values).fillna(0)  # Fill NaN values with zero
    mean = round(stat_values.mean(), 2)
    std_dev = round(stat_values.std(), 2)
    cv = round(std_dev / mean, 2) if mean != 0 else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    team_metrics_data = fetch_team_metrics()
    if team_metrics_data.empty:
        print("No team metrics available.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']
        home_team_abbr = game['home_team']
        away_team_abbr = game['away_team']

        # Map the abbreviations to full team names
        home_team = team_abbr_to_full.get(home_team_abbr, home_team_abbr)
        away_team = team_abbr_to_full.get(away_team_abbr, away_team_abbr)

        # Fetch team metrics for both home and away teams
        home_team_metrics = team_metrics_data[team_metrics_data['TEAM_NAME'] == home_team]
        away_team_metrics = team_metrics_data[team_metrics_data['TEAM_NAME'] == away_team]

        # Ensure there is data for the home and away teams
        if home_team_metrics.empty or away_team_metrics.empty:
            print(f"Missing team data for {home_team} or {away_team}")
            continue

        home_team_metrics = home_team_metrics.iloc[0]
        away_team_metrics = away_team_metrics.iloc[0]

        home_team_offrtg = home_team_metrics['E_OFF_RATING']
        away_team_offrtg = away_team_metrics['E_OFF_RATING']
        home_team_defrtg = home_team_metrics['E_DEF_RATING']
        away_team_defrtg = away_team_metrics['E_DEF_RATING']
        home_team_pace = home_team_metrics['E_PACE']
        away_team_pace = away_team_metrics['E_PACE']

        print(f"Processing teams: {home_team} vs {away_team}")

        # Fetch player stats for both teams
        for team_id, team_name, team_offrtg, team_defrtg, team_pace in [(home_team_id, home_team, home_team_offrtg, home_team_defrtg, home_team_pace), (away_team_id, away_team, away_team_offrtg, away_team_defrtg, away_team_pace)]:
            print(f"Processing team: {team_name}")
            try:
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                if team_roster.empty:
                    print(f"No players found for team {team_name}")
                    continue

                for _, player in team_roster.iterrows():
                    try:
                        player_id = player['PLAYER_ID']
                        player_name = player['PLAYER']
                        player_pos = player['POSITION']
                        print(f"Processing player: {player_name} (ID: {player_id})")

                        game_logs = fetch_player_game_logs(player_id)

                        if game_logs.empty:
                            print(f"No game logs available for {player_name}")
                            continue

                        ppg_values = game_logs['PTS']
                        rpg_values = game_logs['REB']
                        apg_values = game_logs['AST']

                        mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                        mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                        mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                        all_players_data.append({
                            'player_name': player_name,
                            'position': player_pos,
                            'team': team_name,
                            'offensive_rating': team_offrtg,
                            'defensive_rating': team_defrtg,
                            'pace': team_pace,
                            'avg_points': mean_ppg,
                            'std_dev_points': std_dev_ppg,
                            'cv_points': cv_ppg,
                            'avg_rebounds': mean_rpg,
                            'std_dev_rebounds': std_dev_rpg,
                            'cv_rebounds': cv_rpg,
                            'avg_assists': mean_apg,
                            'std_dev_assists': std_dev_apg,
                            'cv_assists': cv_apg
                        })
                    except Exception as e:
                        print(f"Error processing player {player_name}: {e}")
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    return pd.DataFrame(all_players_data)

# Display the results
def display_results(players_df, stat_type):
    if players_df.empty:
        print(f"No data available for {stat_type}.")
        return

    print(f"\n{stat_type} stats:")
    stat_columns = [col for col in players_df.columns if stat_type.lower() in col.lower()]
    display(players_df[['player_name', 'team'] + stat_columns].sort_values(by=f'avg_{stat_type.lower()}'))

# Main execution flow
if __name__ == "__main__":
    players_df = analyze_teams_for_today()
    if not players_df.empty:
        display_results(players_df, stat_type='Points')
        display_results(players_df, stat_type='Rebounds')
        display_results(players_df, stat_type='Assists')




Fetching schedule for 2024-12-02 (EST)
Processing teams: Atlanta Hawks vs New Orleans Pelicans
Processing team: Atlanta Hawks
Processing player: Jalen Johnson (ID: 1630552)
Fetching game logs for player ID: 1630552
Processing player: Keaton Wallace (ID: 1630811)
Fetching game logs for player ID: 1630811
Processing player: Seth Lundy (ID: 1641754)
Fetching game logs for player ID: 1641754
No game logs available for Seth Lundy
Processing player: Kobe Bufkin (ID: 1641723)
Fetching game logs for player ID: 1641723
Processing player: Dyson Daniels (ID: 1630700)
Fetching game logs for player ID: 1630700
Processing player: David Roddy (ID: 1631223)
Fetching game logs for player ID: 1631223
Processing player: Zaccharie Risacher (ID: 1642258)
Fetching game logs for player ID: 1642258
Processing player: Trae Young (ID: 1629027)
Fetching game logs for player ID: 1629027
Processing player: De'Andre Hunter (ID: 1629631)
Fetching game logs for player ID: 1629631
Processing player: Bogdan Bogdanović 

,player_name,team,avg_points,std_dev_points,cv_points
92,Maxwell Lewis,Los Angeles Lakers,0.00,0.00,NaN
51,Dariq Whitehead,Brooklyn Nets,0.00,0.00,NaN
72,Joe Ingles,Minnesota Timberwolves,0.00,0.00,NaN
55,Cui Yongxi,Brooklyn Nets,0.25,0.50,2.00
95,Armel Traoré,Los Angeles Lakers,0.50,1.00,2.00
...,...,...,...,...,...
16,Zion Williamson,New Orleans Pelicans,22.67,9.71,0.43
26,Brandon Ingram,New Orleans Pelicans,22.88,7.20,0.31
63,Cam Thomas,Brooklyn Nets,24.71,8.57,0.35
71,Anthony Edwards,Minnesota Timberwolves,27.68,5.31,0.19



Rebounds stats:


,player_name,team,avg_rebounds,std_dev_rebounds,cv_rebounds
55,Cui Yongxi,Brooklyn Nets,0.00,0.00,NaN
14,Dominick Barlow,Atlanta Hawks,0.00,0.00,NaN
59,Jaylen Martin,Brooklyn Nets,0.00,0.00,NaN
67,Terrence Shannon Jr.,Minnesota Timberwolves,0.00,0.00,NaN
87,Bronny James,Los Angeles Lakers,0.17,0.41,2.41
...,...,...,...,...,...
9,Clint Capela,Atlanta Hawks,8.81,3.27,0.37
0,Jalen Johnson,Atlanta Hawks,9.90,3.68,0.37
40,Nikola Vučević,Chicago Bulls,9.95,2.54,0.26
77,Rudy Gobert,Minnesota Timberwolves,11.16,3.79,0.34



Assists stats:


,player_name,team,avg_assists,std_dev_assists,cv_assists
95,Armel Traoré,Los Angeles Lakers,0.00,0.00,NaN
51,Dariq Whitehead,Brooklyn Nets,0.00,0.00,NaN
55,Cui Yongxi,Brooklyn Nets,0.00,0.00,NaN
59,Jaylen Martin,Brooklyn Nets,0.00,0.00,NaN
28,Karlo Matković,New Orleans Pelicans,0.00,0.00,NaN
...,...,...,...,...,...
57,Ben Simmons,Brooklyn Nets,6.56,2.61,0.40
36,Josh Giddey,Chicago Bulls,6.67,2.71,0.41
18,Elfrid Payton,New Orleans Pelicans,7.83,6.74,0.86
93,LeBron James,Los Angeles Lakers,9.35,3.60,0.39


In [ ]:
# Install necessary packages
!pip install nba_api requests

import pandas as pd
import numpy as np
from nba_api.stats.endpoints import (
    playergamelog, commonteamroster, teamestimatedmetrics
)
from nba_api.stats.library.parameters import Season, SeasonType
from datetime import datetime
import pytz
import requests
import time

# Map team abbreviations to full names
team_abbr_to_full = {
    "CHA": "Charlotte Hornets",
    "NYK": "New York Knicks",
    "ATL": "Atlanta Hawks",
    "CLE": "Cleveland Cavaliers",
    "MEM": "Memphis Grizzlies",
    "NOP": "New Orleans Pelicans",
    "BKN": "Brooklyn Nets",
    "ORL": "Orlando Magic",
    # Add all team mappings here...
}

# Fetch today's NBA games from NBA Stats API
def fetch_schedule():
    try:
        # Get the current date in EST
        EST = pytz.timezone('US/Eastern')
        now = datetime.now(EST)
        today = now.strftime('%Y-%m-%d')  # Format the date as YYYY-MM-DD
        print(f"Fetching schedule for {today} (EST)")

        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        data = response.json()

        games_today = []

        # Iterate through each month's schedule to find games for today
        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],  # Home team abbreviation
                        'away_team': game['v']['ta'],  # Away team abbreviation
                        'home_team_id': game['h']['tid'],  # Home team ID
                        'away_team_id': game['v']['tid']   # Away team ID
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics (Offensive/Defensive stats)
def fetch_team_metrics():
    try:
        print("Fetching team metrics...")
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25",
            season_type="Regular Season",
            league_id="00"
        ).get_data_frames()[0]
        return team_metrics
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return pd.DataFrame()

# Fetch player game logs for the current season
def fetch_player_game_logs(player_id, season="2024-25"):
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Fetching game logs for player ID: {player_id}")
            game_logs = playergamelog.PlayerGameLog(
                player_id=player_id,
                season=season,
                season_type_all_star="Regular Season"
            ).get_data_frames()[0]
            return game_logs
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying game logs fetch for player {player_id} (Attempt {attempt + 1})...")
                time.sleep(5)
            else:
                print(f"Failed to fetch game logs for player {player_id}: {e}")
                return pd.DataFrame()

# Calculate per-player stats (Points, Rebounds, Assists)
def calculate_stats(stat_values):
    if len(stat_values) == 0:
        return None, None, None

    stat_values = pd.Series(stat_values).fillna(0)  # Fill NaN values with zero
    mean = round(stat_values.mean(), 2)
    std_dev = round(stat_values.std(), 2)
    cv = round(std_dev / mean, 2) if mean != 0 else None

    return mean, std_dev, cv

# Analyze today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    team_metrics_data = fetch_team_metrics()
    if team_metrics_data.empty:
        print("No team metrics available.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']
        home_team_abbr = game['home_team']
        away_team_abbr = game['away_team']

        # Map the abbreviations to full team names
        home_team = team_abbr_to_full.get(home_team_abbr, home_team_abbr)
        away_team = team_abbr_to_full.get(away_team_abbr, away_team_abbr)

        # Fetch team metrics for both home and away teams
        home_team_metrics = team_metrics_data[team_metrics_data['TEAM_NAME'] == home_team]
        away_team_metrics = team_metrics_data[team_metrics_data['TEAM_NAME'] == away_team]

        # Ensure there is data for the home and away teams
        if home_team_metrics.empty or away_team_metrics.empty:
            print(f"Missing team data for {home_team} or {away_team}")
            continue

        home_team_metrics = home_team_metrics.iloc[0]
        away_team_metrics = away_team_metrics.iloc[0]

        home_team_offrtg = home_team_metrics['E_OFF_RATING']
        away_team_offrtg = away_team_metrics['E_OFF_RATING']
        home_team_defrtg = home_team_metrics['E_DEF_RATING']
        away_team_defrtg = away_team_metrics['E_DEF_RATING']
        home_team_pace = home_team_metrics['E_PACE']
        away_team_pace = away_team_metrics['E_PACE']

        print(f"Processing teams: {home_team} vs {away_team}")

        # Fetch player stats for both teams
        for team_id, team_name, team_offrtg, team_defrtg, team_pace in [(home_team_id, home_team, home_team_offrtg, home_team_defrtg, home_team_pace), (away_team_id, away_team, away_team_offrtg, away_team_defrtg, away_team_pace)]:
            print(f"Processing team: {team_name}")
            try:
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                if team_roster.empty:
                    print(f"No players found for team {team_name}")
                    continue

                for _, player in team_roster.iterrows():
                    try:
                        player_id = player['PLAYER_ID']
                        player_name = player['PLAYER']
                        player_pos = player['POSITION']
                        print(f"Processing player: {player_name} (ID: {player_id})")

                        game_logs = fetch_player_game_logs(player_id)

                        if game_logs.empty:
                            print(f"No game logs available for {player_name}")
                            continue

                        ppg_values = game_logs['PTS']
                        rpg_values = game_logs['REB']
                        apg_values = game_logs['AST']

                        mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(ppg_values)
                        mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(rpg_values)
                        mean_apg, std_dev_apg, cv_apg = calculate_stats(apg_values)

                        all_players_data.append({
                            'player_name': player_name,
                            'position': player_pos,
                            'team': team_name,
                            'offensive_rating': team_offrtg,
                            'defensive_rating': team_defrtg,
                            'pace': team_pace,
                            'avg_points': mean_ppg,
                            'std_dev_points': std_dev_ppg,
                            'cv_points': cv_ppg,
                            'avg_rebounds': mean_rpg,
                            'std_dev_rebounds': std_dev_rpg,
                            'cv_rebounds': cv_rpg,
                            'avg_assists': mean_apg,
                            'std_dev_assists': std_dev_apg,
                            'cv_assists': cv_apg
                        })
                    except Exception as e:
                        print(f"Error processing player {player_name}: {e}")
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    return pd.DataFrame(all_players_data)

# Display the results
def display_results(players_df, stat_type):
    if players_df.empty:
        print(f"No data available for {stat_type}.")
        return

    print(f"\n{stat_type} stats:")
    stat_columns = [col for col in players_df.columns if stat_type.lower() in col.lower()]
    display(players_df[['player_name', 'team'] + stat_columns].sort_values(by=f'avg_{stat_type.lower()}'))

# Main execution flow
if __name__ == "__main__":
    players_df = analyze_teams_for_today()
    if not players_df.empty:
        display_results(players_df, stat_type='Points')
        display_results(players_df, stat_type='Rebounds')
        display_results(players_df, stat_type='Assists')


Fetching schedule for 2024-12-02 (EST)
Fetching team metrics...
Processing teams: Atlanta Hawks vs New Orleans Pelicans
Processing team: Atlanta Hawks
Processing player: Jalen Johnson (ID: 1630552)
Fetching game logs for player ID: 1630552
Processing player: Keaton Wallace (ID: 1630811)
Fetching game logs for player ID: 1630811
Processing player: Seth Lundy (ID: 1641754)
Fetching game logs for player ID: 1641754
No game logs available for Seth Lundy
Processing player: Kobe Bufkin (ID: 1641723)
Fetching game logs for player ID: 1641723
Processing player: Dyson Daniels (ID: 1630700)
Fetching game logs for player ID: 1630700
Processing player: David Roddy (ID: 1631223)
Fetching game logs for player ID: 1631223
Processing player: Zaccharie Risacher (ID: 1642258)
Fetching game logs for player ID: 1642258
Processing player: Trae Young (ID: 1629027)
Fetching game logs for player ID: 1629027
Processing player: De'Andre Hunter (ID: 1629631)
Fetching game logs for player ID: 1629631
Processing p

,player_name,team,avg_points,std_dev_points,cv_points
14,Dominick Barlow,Atlanta Hawks,0.50,0.71,1.42
28,Karlo Matković,New Orleans Pelicans,0.80,1.10,1.38
11,Mouhamed Gueye,Atlanta Hawks,1.00,1.41,1.41
33,Trey Jemison III,New Orleans Pelicans,2.62,2.63,1.00
15,Vít Krejčí,Atlanta Hawks,2.89,3.18,1.10
23,Daniel Theis,New Orleans Pelicans,3.56,2.61,0.73
1,Keaton Wallace,Atlanta Hawks,4.67,4.19,0.90
22,Jamal Cain,New Orleans Pelicans,5.08,4.73,0.93
2,Kobe Bufkin,Atlanta Hawks,5.29,2.93,0.55
25,Antonio Reeves,New Orleans Pelicans,5.33,9.41,1.77



Rebounds stats:


,player_name,team,avg_rebounds,std_dev_rebounds,cv_rebounds
14,Dominick Barlow,Atlanta Hawks,0.00,0.00,NaN
11,Mouhamed Gueye,Atlanta Hawks,1.00,1.41,1.41
13,Garrison Mathews,Atlanta Hawks,1.06,0.94,0.89
15,Vít Krejčí,Atlanta Hawks,1.11,1.54,1.39
25,Antonio Reeves,New Orleans Pelicans,1.33,1.50,1.13
1,Keaton Wallace,Atlanta Hawks,1.50,1.31,0.87
28,Karlo Matković,New Orleans Pelicans,2.00,2.00,1.00
27,Jose Alvarado,New Orleans Pelicans,2.18,2.23,1.02
2,Kobe Bufkin,Atlanta Hawks,2.29,1.50,0.66
12,Larry Nance Jr.,Atlanta Hawks,2.44,1.67,0.68



Assists stats:


,player_name,team,avg_assists,std_dev_assists,cv_assists
11,Mouhamed Gueye,Atlanta Hawks,0.00,0.00,NaN
28,Karlo Matković,New Orleans Pelicans,0.00,0.00,NaN
14,Dominick Barlow,Atlanta Hawks,0.00,0.00,NaN
25,Antonio Reeves,New Orleans Pelicans,0.42,0.90,2.14
33,Trey Jemison III,New Orleans Pelicans,0.62,0.65,1.05
22,Jamal Cain,New Orleans Pelicans,0.69,0.85,1.23
20,Javonte Green,New Orleans Pelicans,0.86,1.11,1.29
17,Herbert Jones,New Orleans Pelicans,1.00,0.82,0.82
15,Vít Krejčí,Atlanta Hawks,1.11,1.17,1.05
32,Jeremiah Robinson-Earl,New Orleans Pelicans,1.14,1.15,1.01


In [ ]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics, leaguedashteamstats, commonteamroster
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Positional mapping
POSITION_MAPPING = {
    "guard": "G",
    "forward": "F",
    "center": "C",
    "guard-forward": "G-F",
    "forward-center": "F-C",
    "forward-guard": "F-G",
    "center-forward": "C-F"
}

# Fetch Defense vs Position metrics
def fetch_dvp(position):
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense="Opponent",
            per_mode_detailed="PerGame",
            player_position_abbreviation_nullable=position
        )
        dvp_df = stats.get_data_frames()[0]
        dvp_df = dvp_df[["TEAM_ID", "OPP_PTS", "OPP_REB", "OPP_AST"]]
        dvp_df.rename(columns={
            "OPP_PTS": f"OPP_PTS_{position}",
            "OPP_REB": f"OPP_REB_{position}",
            "OPP_AST": f"OPP_AST_{position}"
        }, inplace=True)
        return dvp_df
    except Exception as e:
        print(f"Error fetching DvP metrics for position {position}: {e}")
        return pd.DataFrame()

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        # Map abbreviations to full names
        team_name_mapping = {
            "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets", "CHA": "Charlotte Hornets",
            "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers", "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets",
            "DET": "Detroit Pistons", "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
            "LAC": "Los Angeles Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies", "MIA": "Miami Heat",
            "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves", "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
            "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic", "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
            "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings", "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
            "UTA": "Utah Jazz", "WAS": "Washington Wizards"
        }

        # Adjusted metrics dictionary
        adjusted_metrics = {}
        for abbr, full_name in team_name_mapping.items():
            if full_name in metrics_dict:
                adjusted_metrics[abbr] = metrics_dict[full_name]
            else:
                print(f"Warning: Metrics not found for {abbr} ({full_name}).")

        return adjusted_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    dvp_data = {pos: fetch_dvp(pos) for pos in POSITION_MAPPING.values()}
    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    position = player['POSITION'].lower()
                    position_role = POSITION_MAPPING.get(position, None)

                    if position_role not in dvp_data:
                        print(f"DvP data not found for position {position_role}. Skipping player {player_name}.")
                        continue

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    dvp = dvp_data[position_role]
                    adjusted_ppg = adjust_with_dvp(mean_ppg, dvp, team_name)
                    adjusted_rpg = adjust_with_dvp(mean_rpg, dvp, team_name)
                    adjusted_apg = adjust_with_dvp(mean_apg, dvp, team_name)

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'Position': position,
                        'PPG': mean_ppg,
                        'PPG SD': std_dev_ppg,
                        'PPG CV': cv_ppg,
                        'RPG': mean_rpg,
                        'RPG SD': std_dev_rpg,
                        'RPG CV': cv_rpg,
                        'APG': mean_apg,
                        'APG SD': std_dev_apg,
                        'APG CV': cv_apg,
                        'Adjusted PPG': adjusted_ppg,
                        'Adjusted RPG': adjusted_rpg,
                        'Adjusted APG': adjusted_apg
                    })
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    player_stats_df = pd.DataFrame(all_players_data)
    display(player_stats_df)
    return player_stats_df

# Run analysis
player_stats = analyze_teams_for_today()


Error fetching DvP metrics for position G-F: 'resultSet'
Error fetching DvP metrics for position F-C: 'resultSet'
Error fetching DvP metrics for position F-G: 'resultSet'
Error fetching DvP metrics for position C-F: 'resultSet'
DvP data not found for position None. Skipping player Jalen Johnson.
DvP data not found for position None. Skipping player Keaton Wallace.
DvP data not found for position None. Skipping player Seth Lundy.
DvP data not found for position None. Skipping player Kobe Bufkin.
DvP data not found for position None. Skipping player Dyson Daniels.
DvP data not found for position None. Skipping player David Roddy.
DvP data not found for position None. Skipping player Zaccharie Risacher.
DvP data not found for position None. Skipping player Trae Young.
DvP data not found for position None. Skipping player De'Andre Hunter.
DvP data not found for position None. Skipping player Bogdan Bogdanović.
DvP data not found for position None. Skipping player Clint Capela.
DvP data not

""


In [ ]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import TeamVsPlayer
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Function to fetch DvP stats
def fetch_dvp_stats(team_id):
    try:
        # Correcting the season_type parameter
        stats = TeamVsPlayer(team_id=team_id, season_type='Regular Season')
        return stats.get_data_frames()[0]  # return the first dataframe
    except Exception as e:
        print(f"Error fetching DvP stats for team {team_id}: {e}")
        return None

# Function to process player data
def process_players(team):
    team_players = team['players']
    player_stats = []
    for player in team_players:
        player_data = {
            'Player': player['full_name'],
            'Position': player.get('position', 'Unknown')  # Handle missing 'POS' field
        }
        player_stats.append(player_data)
    return player_stats

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        # Map abbreviations to full names
        team_name_mapping = {
            "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets", "CHA": "Charlotte Hornets",
            "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers", "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets",
            "DET": "Detroit Pistons", "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
            "LAC": "Los Angeles Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies", "MIA": "Miami Heat",
            "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves", "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
            "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic", "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
            "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings", "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
            "UTA": "Utah Jazz", "WAS": "Washington Wizards"
        }

        # Adjusted metrics dictionary
        adjusted_metrics = {}
        for abbr, full_name in team_name_mapping.items():
            if full_name in metrics_dict:
                adjusted_metrics[abbr] = metrics_dict[full_name]
            else:
                print(f"Warning: Metrics not found for {abbr} ({full_name}).")

        return adjusted_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_name})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'PPG': mean_ppg,
                        'PPG SD': std_dev_ppg,
                        'PPG CV': cv_ppg,
                        'RPG': mean_rpg,
                        'RPG SD': std_dev_rpg,
                        'RPG CV': cv_rpg,
                        'APG': mean_apg,
                        'APG SD': std_dev_apg,
                        'APG CV': cv_apg,
                        'Adjusted PPG': adjusted_ppg,
                        'Adjusted RPG': adjusted_rpg,
                        'Adjusted APG': adjusted_apg
                    })
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    player_stats_df = pd.DataFrame(all_players_data)
    display(player_stats_df)
    return player_stats_df

# Run analysis
player_stats = analyze_teams_for_today()


Fetching data for Jalen Johnson (ATL)
Fetching game logs for player ID: 1630552
Error processing team ATL: name 'adjust_prediction' is not defined
Fetching data for Zion Williamson (NOP)
Fetching game logs for player ID: 1629627
Error processing team NOP: name 'adjust_prediction' is not defined
Fetching data for Jayson Tatum (BOS)
Fetching game logs for player ID: 1628369
Error processing team BOS: name 'adjust_prediction' is not defined
Fetching data for Josh Richardson (MIA)
Fetching game logs for player ID: 1626196
Error processing team MIA: name 'adjust_prediction' is not defined
Fetching data for Coby White (CHI)
Fetching game logs for player ID: 1629632
Error processing team CHI: name 'adjust_prediction' is not defined
Fetching data for Dariq Whitehead (BKN)
Fetching game logs for player ID: 1641727
Error processing team BKN: name 'adjust_prediction' is not defined
Fetching data for Terrence Shannon Jr. (MIN)
Fetching game logs for player ID: 1630545
Error processing team MIN: na

""


In [ ]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics, playergamelog, commonteamroster
from nba_api.stats.endpoints import TeamVsPlayer
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from IPython.display import display
import time

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season_all=SeasonAll.current_season).get_data_frames()[0]
        return logs
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Calculate statistics
def calculate_stats(series):
    mean = series.mean()
    std_dev = series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust prediction with team metrics
def adjust_prediction(stat, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating):
    pace_factor = team_metrics['E_PACE'] / avg_pace
    offensive_factor = opposing_metrics['E_DEF_RATING'] / avg_def_rating
    return stat * pace_factor * offensive_factor

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        return metrics_dict, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        for team_abbr in [game['home_team'], game['away_team']]:
            try:
                team_id = teams.find_team_by_abbreviation(team_abbr)['id']
                team_roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_abbr})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_abbr], team_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_abbr], team_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_abbr], team_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_abbr,
                        'PPG': mean_ppg,
                        'PPG SD': std_dev_ppg,
                        'PPG CV': cv_ppg,
                        'RPG': mean_rpg,
                        'RPG SD': std_dev_rpg,
                        'RPG CV': cv_rpg,
                        'APG': mean_apg,
                        'APG SD': std_dev_apg,
                        'APG CV': cv_apg,
                        'Adjusted PPG': adjusted_ppg,
                        'Adjusted RPG': adjusted_rpg,
                        'Adjusted APG': adjusted_apg
                    })
            except Exception as e:
                print(f"Error processing team {team_abbr}: {e}")

    player_stats_df = pd.DataFrame(all_players_data)
    display(player_stats_df)
    return player_stats_df

# Run analysis
player_stats = analyze_teams_for_today()


Fetching data for Jalen Johnson (ATL)
Error fetching game logs for player 1630552: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for Keaton Wallace (ATL)
Error fetching game logs for player 1630811: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for Seth Lundy (ATL)
Error fetching game logs for player 1641754: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for Kobe Bufkin (ATL)
Error fetching game logs for player 1641723: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for Dyson Daniels (ATL)
Error fetching game logs for player 1630700: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for David Roddy (ATL)
Error fetching game logs for player 1631223: PlayerGameLog.__init__() got an unexpected keyword argument 'season_all'
Fetching data for Zaccharie Risacher (ATL)
Error fetching game logs fo

""


In [ ]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        # Map abbreviations to full names
        team_name_mapping = {
            "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets", "CHA": "Charlotte Hornets",
            "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers", "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets",
            "DET": "Detroit Pistons", "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
            "LAC": "Los Angeles Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies", "MIA": "Miami Heat",
            "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves", "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
            "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic", "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
            "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings", "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
            "UTA": "Utah Jazz", "WAS": "Washington Wizards"
        }

        # Adjusted metrics dictionary
        adjusted_metrics = {}
        for abbr, full_name in team_name_mapping.items():
            if full_name in metrics_dict:
                adjusted_metrics[abbr] = metrics_dict[full_name]
            else:
                print(f"Warning: Metrics not found for {abbr} ({full_name}).")

        return adjusted_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_name})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'PPG': mean_ppg,
                        'PPG SD': std_dev_ppg,
                        'PPG CV': cv_ppg,
                        'RPG': mean_rpg,
                        'RPG SD': std_dev_rpg,
                        'RPG CV': cv_rpg,
                        'APG': mean_apg,
                        'APG SD': std_dev_apg,
                        'APG CV': cv_apg,
                        'Adjusted PPG': adjusted_ppg,
                        'Adjusted RPG': adjusted_rpg,
                        'Adjusted APG': adjusted_apg
                    })
            except Exception as e:
                print(f"Error processing team {team_name}: {e}")

    player_stats_df = pd.DataFrame(all_players_data)
    display(player_stats_df)
    return player_stats_df

# Run analysis
player_stats = analyze_teams_for_today()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 3.4 MB/s eta 0:00:00
Fetching data for Bilal Coulibaly (WAS)
Error processing team WAS: name 'fetch_player_game_logs' is not defined
Fetching data for Jalen Hood-Schifino (LAL)
Error processing team LAL: name 'fetch_player_game_logs' is not defined
Fetching data for Max Strus (CLE)
Error processing team CLE: name 'fetch_player_game_logs' is not defined
Fetching data for Jalen Johnson (ATL)
Error processing team ATL: name 'fetch_player_game_logs' is not defined
Fetching data for Jordan Clarkson (UTA)
Error processing team UTA: name 'fetch_player_game_logs' is not defined
Fetching data for Terrence Shannon Jr. (MIN)
Error processing team MIN: name 'fetch_player_game_logs' is not defined
Fetching data for Zyon Pullin (MEM)
Error processing team MEM: name 'fetch_player_game_logs' is not defined
Fetching data for Aaron Holiday (HOU)
Error processing team HOU: name 'fetch_player_game_logs' is not defined
Fetching data for Scoot Hende

""
